In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \sigma(x) = \text{sigmoid}(x) = \frac{e^x}{e^x+1} = \frac{1}{1+e^{-x}} $$

$$ \frac{\partial \sigma}{\partial x} = \frac{e^x}{(e^x+1)^2} = \sigma(x)(1-\sigma(x)) $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def _sigmoid_neg(x: tr.Tensor) -> tr.Tensor:
    """Numerically stable `sigmoid` for negative inputs."""

    exp = tr.exp(x)
    y = exp / (exp + 1)
    return y


def _sigmoid_pos(x: tr.Tensor) -> tr.Tensor:
    """Numerically stable `sigmoid` for positive inputs."""

    y = 1 / (1 + tr.exp(-x))
    return y


def sigmoid(x: tr.Tensor) -> tr.Tensor:
    """Numerically stable `sigmoid` function."""

    x = tr.as_tensor(x, dtype=tr.float32)

    mask = (x >= 0)
    y = tr.empty_like(x)
    y[mask] = _sigmoid_pos(x[mask])
    y[~mask] = _sigmoid_neg(x[~mask])
    return y


class SigmoidFunction(autograd.Function):
    """Custom autograd function for the `sigmoid` function."""

    @staticmethod
    def forward(ctx, x: tr.Tensor) -> tr.Tensor:
        # For demonstration we save `x`, however saving `y` would be more efficient.
        ctx.save_for_backward(x)

        y = sigmoid(x)  
        return y
    
    @staticmethod
    def backward(ctx, grad_output: tr.Tensor) -> tuple[tr.Tensor, ]:
        (x, ) = ctx.saved_tensors

        y = sigmoid(x)
        df_dx = y * (1 - y)
        assert df_dx.shape == x.shape

        dF_dx = grad_output * df_dx
        assert dF_dx.shape == x.shape

        return (dF_dx, )
    

class Sigmoid(nn.Module):
    """Custom module for the `sigmoid` function."""
    
    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return SigmoidFunction.apply(x)

In [ ]:
# Unit tests

def test_sigmoid_basic() -> None:
    assert sigmoid(1000) == approx(1.0)
    assert sigmoid(-1000) == approx(0.0)
    assert sigmoid(0) == approx(0.5)
    assert sigmoid(1) == approx(0.731, atol=0.001, rtol=0)
    assert sigmoid(-1) == approx(0.269, atol=0.001, rtol=0)


def test_SigmoidFunction_gradcheck(samples) -> None:
    def func(x: tr.Tensor) -> tr.Tensor:
        return SigmoidFunction.apply(x)

    x = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (x, ), eps=0.001, atol=0.001)


def test_Sigmoid_compare(samples) -> None:
    x = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = Sigmoid()(x1)
    F1 = y1.sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = tr.sigmoid(x2)
    F2 = y2.sum()
    F2.backward()

    assert y1 == approx(y2)
    assert x1.grad == approx(x2.grad)


if __name__ == "__main__":
    test_sigmoid_basic()

    test_SigmoidFunction_gradcheck(1)
    test_SigmoidFunction_gradcheck(10)
    test_SigmoidFunction_gradcheck(100)

    test_Sigmoid_compare(1)
    test_Sigmoid_compare(10)
    test_Sigmoid_compare(100)